# N056 · 单调队列与带负数的最短区间

**目标：** 同时维护候选顺序和过期边界。

**先修：** N032, N053, N054。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 滑窗极值；前缀和支配；双端弹出；负数使普通窗口失效。

## 从问题到算法

滑窗最大值保留可能成为未来最大值的索引：一个更早且不更大的值可永久删除，因为它先过期且不占优势。含负数的最短区间不能靠“和太小就右扩”的普通窗口；改看前缀差，在递增前缀队列上分别处理“已经达到阈值”和“被新前缀支配”。

## 最小实现

**本章接口：** `max_sliding_window(nums, k)`、`shortest_subarray(nums, k)`

In [1]:
from collections import deque

def max_sliding_window(nums,k):
    if not nums: return []
    if not 1<=k<=len(nums): raise ValueError('invalid window size')
    queue=deque(); answer=[]
    for i,x in enumerate(nums):
        while queue and queue[0]<=i-k: queue.popleft()
        while queue and nums[queue[-1]]<=x: queue.pop()
        queue.append(i)
        if i>=k-1: answer.append(nums[queue[0]])
    return answer

def shortest_subarray(nums,k):
    prefix=[0]
    for x in nums: prefix.append(prefix[-1]+x)
    queue=deque(); best=len(nums)+1
    for i,value in enumerate(prefix):
        while queue and value-prefix[queue[0]]>=k: best=min(best,i-queue.popleft())
        while queue and prefix[queue[-1]]>=value: queue.pop()
        queue.append(i)
    return best if best<=len(nums) else -1

## 正确性、前提与复杂度

更晚且前缀更小的索引，对任意未来右端都给出不更难满足的和与更短长度，支配旧索引。队首一旦已可行，其最短可用右端已检查，未来只会更长。

**代价：** 两个算法 O(n)时间；窗口最大O(k)辅助空间，前缀最短区间O(n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[2,-1,2]; p=[0]
for x in a: p.append(p[-1]+x)
q=deque(); rows=[]
for i,v in enumerate(p):
    solved=[]
    while q and v-p[q[0]]>=3: solved.append((q[0],i,i-q[0])); q.popleft()
    while q and p[q[-1]]>=v: q.pop()
    q.append(i); rows.append((i,v,list(q),solved))
show_table(['i','P[i]','候选左前缀','本次可行区间'],rows)

i,P[i],候选左前缀,本次可行区间
0,0,[0],[]
1,2,"[0, 1]",[]
2,1,"[0, 2]",[]
3,3,"[2, 3]","[(0, 3, 3)]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert max_sliding_window([1,3,-1,-3,5,3,6,7],3)==[3,3,5,5,6,7]
assert shortest_subarray([2,-1,2],3)==3
assert shortest_subarray([1,2],9)==-1
from itertools import product
for n in range(6):
    for a in product([-1,0,2],repeat=n):
        for k in range(1,5):
            lengths=[r-l for l in range(n) for r in range(l+1,n+1) if sum(a[l:r])>=k]
            assert shortest_subarray(a,k)==(min(lengths) if lengths else -1)
        for k in range(1,n+1): assert max_sliding_window(a,k)==[max(a[i:i+k]) for i in range(n-k+1)]
print('N056: 所有本章断言通过')

N056: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释最短区间中两种弹出条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造负数窗口反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 239. Sliding Window Maximum（canonical）
- 862. Shortest Subarray with Sum at Least K（canonical）：[官方入口](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。